# PRT-DeepONet — Velocity variant (full training pipeline)

Trains the divergence-free velocity DeepONet that predicts the velocity field `(ux, uy)` from a
pore geometry, with a divergence-free penalty of weight **λ = 10**.

**Branch CNN** ← geometry, augmented with the **maximum-inscribed-sphere (MIS)** map (local pore
size) and the **upstream-constrained pore radius map (UPRM)** (non-local, path-dependent
connectivity):  `branch1 = [mask, UPRM, MIS]`.  
**Branch FNN** ← the Reynolds number (per-domain scalar condition).  
**Trunk** ← `(x, y)`, the **squared wall distance `d_w²`**, and the **harmonic pressure-gradient
priors `(∂ₓP, ∂ᵧP)`** predicted by the pressure-component U-Net:  `trunk = [x, y, ∂ₓP, ∂ᵧP, d_w²]`.

Target: the velocity field is standardized per component, `u_std = (u − MU) / SD`; the model is
trained on `(u_std, v_std)` and de-standardized at inference. The released checkpoint is
`../parameters/Velocity.pt` (λ=10). The pressure-component U-Net (`../parameters/Pressure_component_UNet.pt`)
is trained separately — see `PRT-DeepONet_Pressure_component_UNet` notes below / the load notebook.


In [ ]:
# ====== 0. Imports ======
import os, copy, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from scipy.ndimage import distance_transform_edt


In [ ]:
# ====== 1. Reproducibility & Paths ======
import os
nx, ny = 64, 148
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True

SEED     = 42
BATCH    = 25
LR       = 1e-3
EPOCHS   = 1000
PATIENCE = 15
LAMBDA   = 10.0        # divergence-free penalty weight (final = weighted pressure, λ=10)

# All inputs live under one data root — set $PRT_DATA_ROOT or edit DATA_ROOT.
# Expected layout is documented in the README ("Data layout").
DATA_ROOT  = os.environ.get('PRT_DATA_ROOT', '../../data')
DATA_PT    = f'{DATA_ROOT}/velocity/split.pt'            # fixed train/test split + base tensors
VDIR       = f'{DATA_ROOT}/velocity/fields/'             # velocity ground truth  v{idx}.npz
RMAP_CACHE = f'{DATA_ROOT}/velocity/mis_map.npz'         # MIS (maximum-inscribed-sphere) map cache
RMAP_STATS = f'{DATA_ROOT}/velocity/mis_stats.npz'       # MIS z-score stats
GV_STATS   = f'{DATA_ROOT}/velocity/pressure_stats.npz'  # (∂ₓP, ∂ᵧP) max-abs stats
GV_MODEL   = '../parameters/Pressure_component_UNet.pt'     # pressure-component U-Net (shipped weights)
MODEL_OUT  = '../parameters/Velocity.pt'                    # trained output


In [ ]:
# ====== 2. Model Definition (paper-style names) ======
# 2a. Pressure-component U-Net: predicts the harmonic pressure-gradient priors (∂ₓP, ∂ᵧP)
#     from the binary pore mask. Used here only for feature inference (trained separately).
N_BLOCKS, BASE_CH, DEPTH, DROPOUT = 4, 64, 6, 0.1
class ConvBlock(nn.Module):
    def __init__(s, cin, cout, p=DROPOUT):
        super().__init__(); s.conv=nn.Conv2d(cin,cout,3,padding=1); s.drop=nn.Dropout2d(p); s.bn=nn.BatchNorm2d(cout)
    def forward(s, x): return s.bn(s.drop(F.relu(s.conv(x))))
class Container(nn.Module):
    def __init__(s, cin, cout, n=N_BLOCKS, p=DROPOUT):
        super().__init__(); s.net=nn.Sequential(*([ConvBlock(cin,cout,p)]+[ConvBlock(cout,cout,p) for _ in range(n-1)]))
    def forward(s, x): return s.net(x)
class PressureComponentUNet(nn.Module):
    """Symmetric U-Net, in=1 (mask), out=2 (∂ₓP, ∂ᵧP)."""
    def __init__(s, in_ch=1, out_ch=2, base=BASE_CH, depth=DEPTH, n=N_BLOCKS, p=DROPOUT):
        super().__init__(); s.depth=depth; s.enc=nn.ModuleList(); c=in_ch
        for _ in range(depth): s.enc.append(Container(c,base,n,p)); c=base
        s.pool=nn.MaxPool2d(2); s.bottleneck=Container(base,base,n,p)
        s.up=nn.ModuleList([nn.ConvTranspose2d(base,base,2,stride=2) for _ in range(depth)])
        s.dec=nn.ModuleList([Container(base*2,base,n,p) for _ in range(depth)])
        s.head_container=Container(base,base,n,p); s.head=nn.Conv2d(base,out_ch,1)
    def forward(s, x):
        skips=[]
        for d in range(s.depth): x=s.enc[d](x); skips.append(x); x=s.pool(x)
        x=s.bottleneck(x)
        for d in range(s.depth):
            skip=skips[s.depth-1-d]; x=s.up[d](x)
            dh=skip.shape[-2]-x.shape[-2]; dw=skip.shape[-1]-x.shape[-1]
            if dh or dw: x=F.pad(x,[dw//2,dw-dw//2,dh//2,dh-dh//2])
            x=torch.cat([x,skip],1); x=s.dec[d](x)
        return s.head(s.head_container(x))

# 2b. Velocity DeepONet
class CustomCNN(nn.Module):
    def __init__(s, in_channels=1, out_dim=128, num_blocks=6):
        super().__init__(); ch=[in_channels,16,32,64,128,256,512][:num_blocks+1]; L=[]
        for i in range(num_blocks): L+=[nn.Conv2d(ch[i],ch[i+1],3,1,1), nn.SiLU(), nn.AvgPool2d(2)]
        s.features=nn.Sequential(*L); h,w=nx,ny
        for _ in range(num_blocks): h//=2; w//=2
        s.fc=nn.Linear(ch[num_blocks]*h*w, out_dim)
    def forward(s, x): x=s.features(x); return s.fc(x.view(x.size(0),-1))
class ReynoldsMLP(nn.Module):
    def __init__(s, in_dim=1, out_dim=128, hidden=128, layers=3):
        super().__init__(); m=[nn.Linear(in_dim,hidden), nn.SiLU()]
        for _ in range(layers-2): m+=[nn.Linear(hidden,hidden), nn.SiLU()]
        m+=[nn.Linear(hidden,out_dim)]; s.net=nn.Sequential(*m)
    def forward(s, x): return s.net(x)
def make_trunk(in_dim=5, out_dim=128, layers=8, width=128):
    m=[nn.Linear(in_dim,width), nn.SiLU()]
    for _ in range(layers-2): m+=[nn.Linear(width,width), nn.SiLU()]
    m+=[nn.Linear(width,out_dim)]; return nn.Sequential(*m)
class DeepONetVec(nn.Module):
    """branch1=[mask,UPRM,MIS] (CNN) | branch2=[Re] (MLP) | trunk=[x,y,∂ₓP,∂ᵧP,d_w²] -> (ux,uy)."""
    def __init__(s, b1_ch=3, out_dim=128, n_out=2):
        super().__init__(); s.branch1=CustomCNN(b1_ch,out_dim,num_blocks=5); s.branch2=ReynoldsMLP(1,out_dim,128,3)
        s.trunk=make_trunk(5,out_dim,8,128); s.bias=nn.Parameter(torch.zeros(n_out)); s.nx,s.ny=nx,ny
        s.n_out=n_out; s.half=out_dim//n_out
    def forward(s, b1, b2, tr):
        N,Lp,D=tr.shape; t=s.trunk(tr.reshape(-1,D)).view(N,Lp,-1)
        b1o=s.branch1(b1).unsqueeze(1); b2o=s.branch2(b2).unsqueeze(1)
        prod=(b1o*b2o*t).view(N,Lp,s.n_out,s.half).sum(-1)
        return (prod+s.bias).view(N,s.nx,s.ny,s.n_out)


In [ ]:
# ====== 3. Data Loader ======
# Loads the fixed split + base tensors, computes the physics features (MIS, d_w², ∂P via the
# pressure U-Net), and standardizes the velocity targets. Returns train/test TensorDatasets.
def _bid(i): return ((int(i)-1) % 3000) + 1
def nwd(m): return distance_transform_edt(m==1).astype(np.float32)

def load_dataset():
    DS = torch.load(DATA_PT, map_location='cpu', weights_only=False)
    train=list(DS['train']); test=list(DS['test'])
    train_b1=DS['train_b1']; test_b1=DS['test_b1']         # 2ch [mask, UPRM]
    train_b2=DS['train_b2']; test_b2=DS['test_b2']         # Reynolds scalar
    train_tr=DS['train_tr']; test_tr=DS['test_tr']         # 2ch [x, y]
    train_roi=DS['train_roi']; test_roi=DS['test_roi']

    # --- Pressure-component U-Net: predict (∂ₓP, ∂ᵧP) per geometry ---
    gvnet = PressureComponentUNet(1,2).to(device)
    gvnet.load_state_dict(torch.load(GV_MODEL, map_location=device, weights_only=False)); gvnet.eval()
    @torch.no_grad()
    def predict_gradvec(m):
        x=torch.from_numpy((m==1).astype(np.float32)[None,None]).to(device)
        o=gvnet(x).float().cpu().numpy()[0]; gx=o[0]; gy=o[1]; gx[m!=1]=0.0; gy[m!=1]=0.0
        return gx.astype(np.float32), gy.astype(np.float32)

    # --- per-base features: ∂P components, d_w² (EDT²), pore mask ---
    GX={}; GY={}; DW2={}; PORE={}
    for s in train+test:
        b=_bid(s['idx'])
        if b not in GX:
            m=np.asarray(s['m']); gx,gy=predict_gradvec(m); GX[b]=gx; GY[b]=gy
            e=nwd(m); f2=(e*e).astype(np.float32); f2[m!=1]=0.0; DW2[b]=f2; PORE[b]=(m==1)
    tr_bases=sorted({_bid(s['idx']) for s in train})
    allv=np.concatenate([DW2[b][DW2[b]>0] for b in tr_bases]); EMIN=float(allv.min()); EMAX=float(allv.max())
    def _dw2n(b):
        e2=DW2[b]; out=np.zeros_like(e2); p=(e2>0); out[p]=(e2[p]-EMIN)/(EMAX-EMIN); return out.astype(np.float32)

    # --- trunk = [x, y, ∂ₓP, ∂ᵧP, d_w²] ---
    def build_trunk5(tr_tensor, slist):
        N,Lp,_=tr_tensor.shape; out=torch.empty((N,Lp,5),dtype=torch.float32)
        out[:,:,0]=tr_tensor[:,:,0]; out[:,:,1]=tr_tensor[:,:,1]
        for i,s in enumerate(slist):
            b=_bid(s['idx'])
            out[i,:,2]=torch.from_numpy(np.ascontiguousarray(GX[b].reshape(-1)))
            out[i,:,3]=torch.from_numpy(np.ascontiguousarray(GY[b].reshape(-1)))
            out[i,:,4]=torch.from_numpy(np.ascontiguousarray(_dw2n(b).reshape(-1)))
        return out
    train_tr5=build_trunk5(train_tr,train); test_tr5=build_trunk5(test_tr,test)

    # --- branch1 3rd channel: MIS (maximum-inscribed-sphere) map, z-scored ---
    _MC=np.load(RMAP_CACHE); MIS={int(k):_MC[k] for k in _MC.files}
    _MS=np.load(RMAP_STATS); MMU=float(_MS['mu']); MSD=float(_MS['sd'])
    def add_mis(b1_2ch, slist):
        N=b1_2ch.shape[0]; ch=torch.empty((N,1,nx,ny),dtype=torch.float32)
        for i,s in enumerate(slist):
            b=_bid(s['idx']); ch[i,0]=torch.from_numpy(np.ascontiguousarray(((MIS[b]-MMU)/MSD).astype(np.float32)))
        return torch.cat([b1_2ch,ch],dim=1)                # -> [mask, UPRM, MIS]
    train_b1_3=add_mis(train_b1.contiguous().clone(),train); test_b1_3=add_mis(test_b1.contiguous().clone(),test)

    # --- velocity targets, standardized over train ROI cells ---
    def load_uv(idx):
        vel=np.load(VDIR+f'v{idx}.npz')['vel']; half=vel.size//2
        return vel[:half].reshape(nx,ny).astype(np.float32), vel[half:].reshape(nx,ny).astype(np.float32)
    UV={int(s['idx']):load_uv(int(s['idx'])) for s in train+test}
    us=[]; vs=[]
    for s in train:
        roi=(np.asarray(s['roi']).reshape(nx,ny)>0.5) if 'roi' in s else (np.asarray(s['m'])==1)
        u,v=UV[int(s['idx'])]; us.append(u[roi]); vs.append(v[roi])
    us=np.concatenate(us); vs=np.concatenate(vs)
    MU_U,SD_U=float(us.mean()),float(us.std()); MU_V,SD_V=float(vs.mean()),float(vs.std())
    global RV; RV=SD_V/SD_U                                # divergence normalization ratio
    def build_y(slist):
        Y=torch.empty((len(slist),nx,ny,2),dtype=torch.float32)
        for i,s in enumerate(slist):
            u,v=UV[int(s['idx'])]; Y[i,...,0]=torch.from_numpy((u-MU_U)/SD_U); Y[i,...,1]=torch.from_numpy((v-MU_V)/SD_V)
        return Y
    train_yv=build_y(train); test_yv=build_y(test)

    # --- interior pore mask (for the divergence penalty) ---
    def interior_mask(slist):
        IM=torch.zeros((len(slist),nx,ny),dtype=torch.bool)
        for i,s in enumerate(slist):
            p=PORE[_bid(s['idx'])]; inter=np.zeros_like(p)
            inter[1:-1,1:-1]=p[1:-1,1:-1]&p[1:-1,2:]&p[1:-1,:-2]&p[2:,1:-1]&p[:-2,1:-1]
            IM[i]=torch.from_numpy(inter)
        return IM
    train_im=interior_mask(train); test_im=interior_mask(test)

    stats=dict(MU_U=MU_U,SD_U=SD_U,MU_V=MU_V,SD_V=SD_V,RV=RV,EMIN=EMIN,EMAX=EMAX)
    train_ds=TensorDataset(train_b1_3,train_b2,train_tr5,train_yv,train_roi,train_im)
    test_ds =TensorDataset(test_b1_3, test_b2, test_tr5, test_yv, test_roi)
    return train_ds, test_ds, stats


In [ ]:
# ====== 4. Training Utilities ======
class ROIHuberLoss(nn.Module):
    """Huber loss masked to the porous ROI."""
    def __init__(s, delta=1.0): super().__init__(); s.delta=delta
    def forward(s, pred, target, roi):
        mask=(roi>0.5).unsqueeze(-1); diff=(pred-target)*mask; absd=torch.abs(diff)
        quad=0.5*absd.pow(2); lin=s.delta*(absd-0.5*s.delta); loss=torch.where(absd<=s.delta,quad,lin)
        return loss.sum()/(mask.sum().clamp(min=1.0)*pred.shape[-1])

def div_norm_loss(pred, interior):
    """Divergence-free penalty in standardized units: du_std/dj + RV*dv_std/di (central diff),
    averaged over interior pore cells."""
    u=pred[...,0]; v=pred[...,1]
    dudj=torch.zeros_like(u); dvdi=torch.zeros_like(v)
    dudj[:,1:-1,1:-1]=(u[:,1:-1,2:]-u[:,1:-1,:-2])*0.5
    dvdi[:,1:-1,1:-1]=(v[:,2:,1:-1]-v[:,:-2,1:-1])*0.5
    divn=dudj+RV*dvdi; m=interior.float()
    return (divn.pow(2)*m).sum()/m.sum().clamp(min=1.0)

def train_model(model, train_ds, test_ds, lam=LAMBDA, num_epochs=EPOCHS, lr=LR, batch_size=BATCH, patience=PATIENCE):
    tl=DataLoader(train_ds,batch_size=batch_size,shuffle=True); el=DataLoader(test_ds,batch_size=50)
    crit=ROIHuberLoss(1.0); opt=torch.optim.AdamW(model.parameters(),lr=lr)
    scaler=torch.amp.GradScaler('cuda',enabled=(device.type=='cuda'))
    best=copy.deepcopy(model.state_dict()); bv=1e9; bep=0; ni=0; t1=time.time()
    for ep in range(1,num_epochs+1):
        model.train()
        for _b1,_b2,_t,_y,_rm,_im in tl:
            _b1=_b1.to(device);_b2=_b2.to(device);_t=_t.to(device);_y=_y.to(device);_rm=_rm.to(device);_im=_im.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda',enabled=(device.type=='cuda')):
                pred=model(_b1,_b2,_t)
                lh=crit(pred,_y,_rm)
                ld=div_norm_loss(pred,_im) if lam>0 else torch.zeros((),device=device)
                loss=lh+lam*ld
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        # early stop on validation data-fidelity (Huber) only
        model.eval(); tot=0.0; n=0
        with torch.no_grad():
            for _b1,_b2,_t,_y,_rm in el:
                with torch.amp.autocast('cuda',enabled=(device.type=='cuda')):
                    tot+=float(crit(model(_b1.to(device),_b2.to(device),_t.to(device)),_y.to(device),_rm.to(device)).item()); n+=1
        vl=tot/n
        if vl<bv-1e-6: bv=vl; bep=ep; ni=0; best=copy.deepcopy(model.state_dict())
        else:
            ni+=1
            if ni>=patience: break
        if ep%10==0 or ep==1: print(f'  ep{ep:3d} valHuber{vl:.5f} best{bv:.5f}@{bep} {time.time()-t1:.0f}s', flush=True)
    model.load_state_dict(best); return bep, bv


In [ ]:
# ====== 5. Evaluation Example (velocity NRMSE over the ROI, physical units) ======
@torch.no_grad()
def evaluate(model, test_ds, stats, num_samples=5):
    SD_U,MU_U,SD_V,MU_V=stats['SD_U'],stats['MU_U'],stats['SD_V'],stats['MU_V']
    model.eval()
    for i in range(min(num_samples,len(test_ds))):
        b1,b2,tr,y,roi,im=test_ds[i]
        pred=model(b1[None].to(device),b2[None].to(device),tr[None].to(device))[0].cpu().numpy()
        gt=y.numpy(); m=(roi.numpy().reshape(nx,ny)>0.5)
        pu=pred[...,0]*SD_U+MU_U; pv=pred[...,1]*SD_V+MU_V
        gu=gt[...,0]*SD_U+MU_U;   gv=gt[...,1]*SD_V+MU_V
        pmag=np.sqrt(pu**2+pv**2); gmag=np.sqrt(gu**2+gv**2)
        nrmse=np.sqrt(np.mean((pmag[m]-gmag[m])**2))/(gmag[m].max()-gmag[m].min()+1e-12)
        print(f'  sample {i}: |vel| NRMSE = {nrmse:.4f}')


In [ ]:
# ====== 6. Main Entry ======
if __name__ == '__main__':
    np.random.seed(SEED); torch.manual_seed(SEED)
    if device.type=='cuda': torch.cuda.manual_seed_all(SEED)
    # 1) Load data + build features
    train_ds, test_ds, stats = load_dataset()
    print('train/test:', len(train_ds), len(test_ds), '| stats:', {k:round(v,5) for k,v in stats.items()})
    # 2) Build model:  branch1=[mask,UPRM,MIS] | branch2=[Re] | trunk=[x,y,∂ₓP,∂ᵧP,d_w²]
    model = DeepONetVec(b1_ch=3, out_dim=128, n_out=2).to(device)
    # 3) Train (Huber over ROI + λ·divergence-free penalty, λ=10)
    bep, bv = train_model(model, train_ds, test_ds, lam=LAMBDA)
    print(f'best epoch {bep}, val Huber {bv:.5f}')
    # 4) Evaluate
    evaluate(model, test_ds, stats)
    # 5) Save the trained parameters (state_dict) -> released as ../parameters/Velocity.pt
    torch.save(model.state_dict(), MODEL_OUT)
    print('[save]', MODEL_OUT)
